# 11장. LLM과 함께 분석 질문을 다듬기

이 노트북은 `book/chapters/ch11_llm_prompt_analysis.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 LLM에게 원본 데이터를 그대로 맡기는 것이 아니라, **안전한 데이터 구조 요약과 검증 가능한 프롬프트**를 만들어 분석 보조 도구로 활용하는 것입니다.

주의: 현재 노트북 파일명은 기존 목차 기준의 `ch11_insight_generation.ipynb`이지만, 실제 11장 강의안 내용은 LLM 프롬프트 기반 분석 보조입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 가능하면 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없으면 `data/raw/*.csv`를 사용합니다.
- 원본 고객명, 연락처, 개별 거래 상세는 LLM에 직접 입력하지 않는다는 원칙을 확인합니다.
- 데이터 구조 요약, 컬럼 요약, 프롬프트 템플릿, 검증 체크리스트, 프롬프트 로그를 `reports/`에 저장합니다.


## 1. LLM은 분석가를 대체하지 않는다

LLM은 분석 질문을 정리하고, 코드 초안을 만들고, 결과 해석 문장을 다듬는 데 도움을 줄 수 있습니다. 하지만 최종 판단과 검증은 사람이 해야 합니다.

| 분석 단계 | LLM이 도와줄 수 있는 일 | 사람이 반드시 확인할 일 |
|---|---|---|
| 데이터 구조 이해 | 확인할 항목 제안 | 실제 컬럼명과 타입 확인 |
| 분석 질문 만들기 | 질문 후보 제안 | 현재 데이터로 답할 수 있는지 검토 |
| 전처리 | 코드 초안 작성 | 처리 기준이 적절한지 판단 |
| 시각화 | 그래프 추천 | 축과 그래프 종류 검증 |
| 머신러닝 | 모델링 코드 초안 | 데이터 누수와 평가 지표 검토 |
| 결과 해석 | 문장 초안 작성 | 원인 단정과 과장 표현 수정 |


## 2. 패키지와 경로 설정

노트북 실행 위치가 프로젝트 루트인지 `notebooks/` 폴더인지에 따라 경로를 자동으로 맞춥니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('원본 데이터 폴더:', RAW_DIR)
print('보고서 폴더:', REPORT_DIR)


## 3. 데이터 불러오기

LLM에게 원본 데이터를 직접 넣기 전에, 먼저 우리가 데이터 구조를 요약합니다. 전처리 데이터가 있으면 우선 사용하고, 없으면 원본 데이터를 사용합니다.


In [ ]:
processed_files = {
    'customers': PROCESSED_DIR / 'customers_clean.csv',
    'products': PROCESSED_DIR / 'products_clean.csv',
    'orders': PROCESSED_DIR / 'orders_clean.csv',
    'order_items': PROCESSED_DIR / 'order_items_clean.csv',
}

raw_files = {
    'customers': RAW_DIR / 'customers.csv',
    'products': RAW_DIR / 'products.csv',
    'orders': RAW_DIR / 'orders.csv',
    'order_items': RAW_DIR / 'order_items.csv',
}

if all(path.exists() for path in processed_files.values()):
    selected_files = processed_files
    print('전처리 데이터를 사용합니다.')
elif all(path.exists() for path in raw_files.values()):
    selected_files = raw_files
    print('원본 데이터를 사용합니다.')
else:
    raise FileNotFoundError('판매 데이터 파일이 없습니다. 먼저 python scripts/preprocess_data.py 또는 python scripts/generate_sample_data.py 를 실행하세요.')

datasets = {name: pd.read_csv(path) for name, path in selected_files.items()}

for name, df in datasets.items():
    print(name, df.shape)


## 4. LLM 입력용 데이터 구조 요약 만들기

LLM에는 원본 데이터 전체가 아니라 데이터셋 이름, 행/열 수, 컬럼명, 결측치 수 같은 구조 정보를 제공하는 것이 안전합니다.


In [ ]:
dataset_descriptions = {
    'customers': '고객 정보',
    'products': '상품 정보',
    'orders': '주문 정보',
    'order_items': '주문 상세 정보',
}

dataset_summary_rows = []
for name, df in datasets.items():
    dataset_summary_rows.append({
        'dataset': name,
        'description': dataset_descriptions.get(name, ''),
        'rows': df.shape[0],
        'columns': df.shape[1],
        'column_list': ', '.join(df.columns),
        'missing_values': int(df.isna().sum().sum()),
        'duplicated_rows': int(df.duplicated().sum()),
    })

dataset_summary = pd.DataFrame(dataset_summary_rows)
dataset_summary.to_csv(REPORT_DIR / 'ch11_dataset_summary_for_llm.csv', index=False, encoding='utf-8-sig')
dataset_summary


In [ ]:
column_summary_rows = []
for name, df in datasets.items():
    for col in df.columns:
        column_summary_rows.append({
            'dataset': name,
            'column': col,
            'dtype': str(df[col].dtype),
            'missing_count': int(df[col].isna().sum()),
            'unique_count': int(df[col].nunique(dropna=True)),
            'example_values': ', '.join(map(str, df[col].dropna().astype(str).unique()[:3])),
        })

column_summary = pd.DataFrame(column_summary_rows)
column_summary.to_csv(REPORT_DIR / 'ch11_column_summary_for_llm.csv', index=False, encoding='utf-8-sig')
column_summary.head(20)


## 5. 안전한 LLM 입력 문맥 만들기

아래 텍스트는 LLM에 붙여 넣어도 비교적 안전한 데이터 구조 설명입니다. 원본 개인정보나 개별 거래 상세는 포함하지 않습니다.


In [ ]:
dataset_lines = []
for _, row in dataset_summary.iterrows():
    dataset_lines.append(
        f"- {row['dataset']} ({row['description']}): {row['rows']}행, {row['columns']}열, 컬럼: {row['column_list']}"
    )

safe_context_text = '\n'.join([
    '# LLM 입력용 데이터 구조 요약',
    '',
    '## 데이터셋 개요',
    *dataset_lines,
    '',
    '## 주의',
    '- 원본 고객명, 이메일, 전화번호, 주소, 개별 거래 상세는 입력하지 않습니다.',
    '- 아래 정보는 데이터의 구조와 분석 목적을 설명하기 위한 요약 정보입니다.',
])

safe_context_path = REPORT_DIR / 'ch11_safe_llm_context.md'
safe_context_path.write_text(safe_context_text, encoding='utf-8')
print(safe_context_text)


## 6. 좋은 프롬프트의 구조

좋은 프롬프트는 배경, 데이터 구조, 요청 작업, 제약 조건, 출력 형식, 검증 요청을 함께 포함합니다.

| 구성 요소 | 설명 | 예시 |
|---|---|---|
| 역할 | 어떤 관점으로 답할지 지정 | Python 데이터 분석 멘토 |
| 목적 | 무엇을 하려는지 설명 | 카테고리별 매출 분석 |
| 데이터 구조 | 데이터셋과 컬럼 정보 제공 | order_items, products 컬럼 목록 |
| 요청 작업 | 작성할 코드나 해석 작업 명시 | merge 후 groupby 집계 |
| 제약 조건 | 추측 금지, 컬럼명 생성 금지 | 실제 데이터에 없는 컬럼명을 만들지 말 것 |
| 출력 형식 | 원하는 답변 형태 지정 | 코드, 설명, 검증 항목 |
| 검증 요청 | 확인해야 할 위험 요소 포함 | 병합 전후 행 수 확인 코드 포함 |


## 7. 분석 단계별 프롬프트 템플릿 만들기

분석 질문 생성, 전처리 계획, 시각화, 회귀, 분류, 결과 해석에 사용할 수 있는 프롬프트 템플릿을 표로 정리합니다.


In [ ]:
prompt_templates = pd.DataFrame([
    {
        'step': '분석 질문 생성',
        'purpose': '현재 데이터로 가능한 분석 질문 만들기',
        'prompt': '''온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋 구조:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 분석 가능한 질문 10개를 제안해 주세요.
2. 각 질문에 필요한 데이터셋과 컬럼을 함께 적어 주세요.
3. 집계, 시각화, 회귀, 분류 중 어떤 방식으로 접근할 수 있는지 표시해 주세요.
4. 현재 데이터로 답할 수 없는 질문은 제외하거나 추가 데이터가 필요하다고 표시해 주세요.

주의:
- 실제 데이터에 없는 컬럼을 만들지 마세요.
- 고객 선호도, 광고 효과, 프로모션 효과처럼 현재 데이터에 없는 원인을 단정하지 마세요.''',
        'validation_point': '현재 데이터로 답할 수 있는 질문인지 확인',
    },
    {
        'step': '전처리 계획',
        'purpose': '결측치, 중복, 타입 문제를 확인하는 계획 만들기',
        'prompt': '''다음 온라인 쇼핑몰 데이터의 전처리 계획을 세우려고 합니다.

요청:
1. 각 데이터셋에서 확인해야 할 결측치, 중복, 데이터 타입 문제를 정리해 주세요.
2. order_date와 signup_date를 날짜형으로 변환할 때 확인할 사항을 알려 주세요.
3. 문자열 범주값의 표기 차이를 확인하는 코드를 제안해 주세요.
4. 전처리 후 저장할 파일명을 제안해 주세요.

주의:
- 결측치나 이상값을 무조건 삭제하지 마세요.
- 실제 데이터에 없는 컬럼명을 만들지 마세요.''',
        'validation_point': '무조건 삭제나 대체를 제안하지 않았는지 확인',
    },
    {
        'step': '시각화',
        'purpose': '분석 질문에 맞는 그래프 선택과 코드 초안 만들기',
        'prompt': '''온라인 쇼핑몰 데이터 분석 결과를 시각화하려고 합니다.

분석 질문:
1. 카테고리별 매출은 어떻게 다른가?
2. 월별 매출은 어떻게 변하는가?
3. 상품 가격은 어떤 구간에 몰려 있는가?
4. 상품 가격과 판매 수량은 관계가 있는가?
5. 구매 금액 상위 고객은 누구인가?

요청:
1. 각 질문에 적합한 그래프 종류를 추천해 주세요.
2. 그래프를 선택한 이유를 설명해 주세요.
3. 그래프 해석 시 주의할 점을 알려 주세요.

주의:
- 월별 매출을 파이 차트로 추천하지 마세요.
- 고객명이 포함되는 그래프는 익명화 필요성을 언급해 주세요.''',
        'validation_point': '그래프 종류가 분석 질문과 맞는지 확인',
    },
    {
        'step': '회귀 모델링',
        'purpose': '주문별 총금액 예측 코드 초안 만들기',
        'prompt': '''온라인 쇼핑몰 주문 데이터를 사용해 주문별 총금액을 예측하는 회귀 모델을 만들려고 합니다.

예측 대상:
- order_total: 주문별 line_total 합계

요청:
1. 주문별 모델링 데이터셋을 만드는 pandas 코드를 작성해 주세요.
2. train/test split을 적용해 주세요.
3. LinearRegression과 RandomForestRegressor를 비교해 주세요.
4. MAE, RMSE, R2를 계산해 주세요.
5. 데이터 누수가 발생할 수 있는 부분을 설명해 주세요.

주의:
- order_total을 입력값으로 사용하지 마세요.
- 실제 데이터에 없는 컬럼명을 만들지 마세요.''',
        'validation_point': '예측 대상이 입력값에 섞이지 않았는지 확인',
    },
    {
        'step': '분류 모델링',
        'purpose': '주문 취소 여부 예측 코드 초안 만들기',
        'prompt': '''온라인 쇼핑몰 주문 데이터를 사용해 주문 취소 여부를 예측하는 분류 모델을 만들려고 합니다.

예측 대상:
- is_cancelled: order_status가 cancelled이면 1, 아니면 0

요청:
1. 주문별 분류 데이터셋을 만드는 pandas 코드를 작성해 주세요.
2. LogisticRegression과 RandomForestClassifier를 비교해 주세요.
3. accuracy, precision, recall, confusion matrix를 계산해 주세요.

주의:
- order_status 원본 컬럼을 입력값으로 사용하지 마세요.
- 실제 데이터에 없는 컬럼명을 만들지 마세요.''',
        'validation_point': 'order_status가 입력값에 포함되지 않았는지 확인',
    },
    {
        'step': '결과 해석',
        'purpose': '분석 결과를 관찰, 가설, 추가 질문으로 정리하기',
        'prompt': '''다음은 카테고리별 매출 분석 결과입니다.

요청:
1. 데이터로 확인 가능한 관찰 내용을 작성해 주세요.
2. 가능한 원인 가설을 조심스럽게 작성해 주세요.
3. 추가로 확인해야 할 분석 질문을 제안해 주세요.
4. 보고서에 넣을 수 있는 문장으로 정리해 주세요.

조건:
- 고객 선호, 프로모션 효과 같은 원인을 단정하지 마세요.
- 데이터에 없는 내용을 추측하지 마세요.
- 관찰과 가설을 구분해 주세요.''',
        'validation_point': '원인 단정과 과장 표현 확인',
    },
])

prompt_templates.to_csv(REPORT_DIR / 'ch11_prompt_templates.csv', index=False, encoding='utf-8-sig')
prompt_templates[['step', 'purpose', 'validation_point']]


## 8. LLM 답변 검증 체크리스트 만들기

LLM 답변은 반드시 사람이 검증해야 합니다. 코드가 실행되어도 컬럼명, 병합 기준, 데이터 누수, 해석 문장이 틀릴 수 있습니다.


In [ ]:
llm_review_checklist = pd.DataFrame({
    'check_item': [
        '원본 개인정보나 거래 상세를 입력하지 않았는가?',
        '데이터 구조 요약만 입력했는가?',
        '분석 목적을 명확히 작성했는가?',
        '원하는 출력 형식을 지정했는가?',
        '실제 데이터에 없는 컬럼명을 만들지 말라고 요청했는가?',
        'LLM이 만든 코드가 실제로 실행되는가?',
        '컬럼명과 데이터 타입이 실제 데이터와 일치하는가?',
        '병합 기준이 올바른가?',
        '날짜 변환과 결측치 확인 코드가 포함되었는가?',
        '머신러닝 코드에서 데이터 누수가 없는가?',
        '평가 지표가 문제 유형에 맞는가?',
        '해석 문장에서 원인을 단정하지 않았는가?',
        '데이터에 없는 내용을 추측하지 않았는가?',
        'LLM 답변을 수정한 내용을 기록했는가?',
    ],
    'result': ['□'] * 14,
    'memo': [''] * 14,
})

llm_review_checklist.to_csv(REPORT_DIR / 'ch11_llm_review_checklist.csv', index=False, encoding='utf-8-sig')
llm_review_checklist


## 9. 프롬프트 사용 로그 만들기

LLM을 분석에 사용했다면 어떤 질문을 입력했고, 어떤 답변을 참고했으며, 무엇을 수정했는지 기록해야 합니다. 이것은 분석의 재현성과 신뢰성을 높입니다.


In [ ]:
llm_usage_log = pd.DataFrame({
    'step': [
        '데이터 구조 설명',
        '분석 질문 생성',
        '전처리 계획',
        '시각화 코드 초안',
        '회귀 모델링 코드 초안',
        '분류 모델링 코드 초안',
        '결과 해석 문장 작성',
    ],
    'purpose': [
        '데이터셋 구조를 설명하고 분석 전 확인 사항 정리',
        '현재 데이터로 가능한 분석 질문 생성',
        '결측치, 중복, 날짜 처리 계획 수립',
        '분석 질문에 맞는 그래프와 matplotlib 코드 초안 생성',
        '주문별 총금액 예측 모델 코드 초안 생성',
        '주문 취소 여부 예측 모델 코드 초안 생성',
        '집계 결과를 보고서 문장으로 정리',
    ],
    'input_summary': ['데이터 구조 요약'] * 7,
    'llm_answer_summary': [''] * 7,
    'validation_point': [
        '실제 컬럼명과 데이터 타입 확인',
        '현재 데이터로 답할 수 있는 질문인지 확인',
        '무조건 삭제나 대체를 제안하지 않았는지 확인',
        '그래프 종류가 분석 질문과 맞는지 확인',
        '데이터 누수와 평가 지표 확인',
        '정답 컬럼이 입력값에 섞이지 않았는지 확인',
        '원인 단정과 과장 표현 확인',
    ],
    'revision_note': [''] * 7,
    'final_use': ['부분 사용'] * 7,
})

llm_usage_log.to_csv(REPORT_DIR / 'ch11_llm_usage_log.csv', index=False, encoding='utf-8-sig')
llm_usage_log


## 10. Markdown 프롬프트 로그 저장하기

CSV뿐 아니라 Markdown 형태의 프롬프트 로그를 남기면 보고서나 GitHub 문서에 바로 활용할 수 있습니다.


In [ ]:
prompt_log_text = f'''# Chapter 11 LLM 프롬프트 로그

## 1. 사용 목적

LLM을 활용해 데이터 구조 설명, 분석 질문 생성, 전처리 계획, 시각화 코드, 머신러닝 코드, 결과 해석 문장의 초안을 만들고 검증했습니다.

## 2. 사용 원칙

- 원본 개인정보와 거래 상세 데이터는 입력하지 않았습니다.
- 컬럼명, 데이터 구조, 집계 결과 중심으로 질문했습니다.
- LLM 답변은 실제 코드 실행과 결과 비교를 통해 검증했습니다.
- 데이터에 없는 원인을 단정하는 문장은 수정했습니다.

## 3. 사용 로그 템플릿

```text
{llm_usage_log.to_string(index=False)}
```

## 4. 검증 체크리스트

```text
{llm_review_checklist.to_string(index=False)}
```

## 5. 검증 기준

- 실제 컬럼명과 일치하는가?
- 병합 기준이 올바른가?
- 날짜 변환과 결측치 확인이 포함되었는가?
- 머신러닝 코드에서 데이터 누수가 없는가?
- 해석 문장이 데이터에 근거하는가?
'''

prompt_log_path = REPORT_DIR / 'ch11_llm_prompt_log.md'
prompt_log_path.write_text(prompt_log_text, encoding='utf-8')
print('프롬프트 로그 저장 완료:', prompt_log_path)


## 11. 소스 모듈로 전체 자료 생성하기

위에서 단계별로 만든 LLM 프롬프트 보조 자료는 `src/llm_prompt_analysis.py`에 함수로 정리되어 있습니다. 전체 파이프라인을 한 번에 실행할 수 있습니다.


In [ ]:
from src.llm_prompt_analysis import run_llm_prompt_analysis

llm_prompt_result = run_llm_prompt_analysis(
    processed_dir=PROCESSED_DIR,
    raw_dir=RAW_DIR,
    report_dir=REPORT_DIR,
)

llm_prompt_result['prompt_templates'][['step', 'purpose', 'validation_point']]


## 12. 스크립트로 한 번에 실행하기

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행하면 11장 LLM 프롬프트 보조 자료가 자동으로 생성됩니다.

```bash
python scripts/run_llm_prompt_analysis.py
```


## 13. 실습 과제

아래 과제를 직접 해결해 보세요.

1. `ch11_safe_llm_context.md` 내용을 LLM에 붙여 넣고 분석 질문 10개를 요청하세요.
2. LLM이 제안한 질문 중 현재 데이터로 답할 수 없는 질문을 찾아 수정하세요.
3. 전처리 프롬프트를 작성하고, LLM이 무조건 삭제를 제안하는지 확인하세요.
4. 시각화 프롬프트를 작성하고, 그래프 선택이 질문과 맞는지 검토하세요.
5. 회귀 또는 분류 프롬프트를 작성하고, 데이터 누수 가능성을 검토하세요.
6. `ch11_llm_usage_log.csv`에 실제 사용한 프롬프트와 수정 내용을 기록하세요.


In [ ]:
# 과제 1. prompt_templates에서 하나를 골라 실제 LLM 질문용 텍스트로 복사해 보세요.
# 예시: print(prompt_templates.loc[0, 'prompt'])


## 14. 정리

이번 장에서는 다음 내용을 실습했습니다.

- LLM을 분석 보조 도구로 사용하는 원칙
- 원본 데이터 대신 데이터 구조 요약 제공
- 데이터셋 요약표와 컬럼 요약표 생성
- 안전한 LLM 입력 문맥 작성
- 분석 질문, 전처리, 시각화, 회귀, 분류, 해석 프롬프트 템플릿 작성
- LLM 답변 검증 체크리스트 작성
- 프롬프트 사용 로그 작성
- `src/llm_prompt_analysis.py`와 `scripts/run_llm_prompt_analysis.py`로 반복 실행 가능한 구조 만들기

다음 장에서는 LLM이 생성한 분석 코드와 결과를 더 구체적으로 검토합니다.
